# BÀI TẬP: MEDICAL INSURANCE COST
**Nguồn:** kaggle.com/datasets/mosapabdelghany/medical-insurance-cost-dataset



## Setup

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')


csv_path = 'https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv'

df = pd.read_csv(csv_path)
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [3]:
# TODO
print("so dong:", df.shape[0], "so cot:", df.shape[1])
print()
print(df.info())

so dong: 1338 so cot: 7

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB
None


## A.2. Missing values & Duplicate data

In [8]:
# TODO
missing = df.isnull().sum()
print("Missing values:\n", missing)
print("\nDuplicate data:", df.duplicated().sum())

Missing values:
 age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

Duplicate data: 1


## A.3. Invalid values

In [ ]:
# TODO
print("Kiem tra gia tri invalid")
print((df[['age', 'bmi', 'children', 'charges']] < 0).sum())
print("\nUnique values ở các cột phân loại:")
for col in ['sex', 'smoker', 'region']:
    print(f"{col}: {df[col].unique()}")

Kiểm tra giá trị âm hoặc bất thường:
age         0
bmi         0
children    0
charges     0
dtype: int64

Unique values ở các cột phân loại:
sex: <StringArray>
['female', 'male']
Length: 2, dtype: str
smoker: <StringArray>
['yes', 'no']
Length: 2, dtype: str
region: <StringArray>
['southwest', 'southeast', 'northwest', 'northeast']
Length: 4, dtype: str


## A.4. Create a new column
Tạo cột `bmi_group`: Normal (<25), Overweight (25-30), Obese (>=30).

In [17]:
conditions = [df['bmi'] < 25, df['bmi'] < 30]
choices = ['Normal', 'Overweight']

df['bmi_group'] = np.select(conditions, choices, default='Obese')
df

,age,sex,bmi,children,smoker,region,charges,bmi_group
0,19,female,27.900,0,yes,southwest,16884.92400,Overweight
1,18,male,33.770,1,no,southeast,1725.55230,Obese
2,28,male,33.000,3,no,southeast,4449.46200,Obese
3,33,male,22.705,0,no,northwest,21984.47061,Normal
4,32,male,28.880,0,no,northwest,3866.85520,Overweight
...,...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830,Obese
1334,18,female,31.920,0,no,northeast,2205.98080,Obese
1335,18,female,36.850,0,no,southeast,1629.83350,Obese
1336,21,female,25.800,0,no,southwest,2007.94500,Overweight


---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [22]:
# # TODO
# num_cols = ['age', 'bmi', 'children', 'charges']
# df[num_cols].agg(['mean', 'median', lambda x: stats.mode(x, keepdims=False)[0]]).rename(index={'<lambda>': 'mode'})

cols = ['age', 'bmi', 'children', 'charges']
for col in cols:
    _mean = df[col].mean()
    _median = df[col].median()
    _mode = df[col].mode()[0]
    print(f"[{col}] Mean: {_mean: .2f} | Median: {_median: .2f} | Mode: {_mode: .2f}")

[age] Mean:  39.21 | Median:  39.00 | Mode:  18.00
[bmi] Mean:  30.66 | Median:  30.40 | Mode:  32.30
[children] Mean:  1.09 | Median:  1.00 | Mode:  0.00
[charges] Mean:  13270.42 | Median:  9382.03 | Mode:  1639.56


## Group 2 — Dispersion

In [ ]:
# TODO
def calc_range(x): return x.max() - x.min()
def calc_iqr(x): return x.quantile(0.75) - x.quantile(0.25)

disp = df[num_cols].agg(['std', 'var', calc_range, calc_iqr])
disp.index = ['std', 'var', 'range', 'iqr']
disp

## Group 3 — Location and Shape

In [ ]:
# TODO
percentiles = df[num_cols].quantile([0.25, 0.50, 0.75])
skewness = df[num_cols].skew().to_frame(name='skewness').T
kurt = df[num_cols].kurtosis().to_frame(name='kurtosis').T
pd.concat([percentiles, skewness, kurt])

---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Người hút thuốc trả chi phí cao hơn bao nhiêu lần so với người không hút, và có đồng đều giữa các vùng không?

In [ ]:
# TODO
smoker_region = df.groupby(['region', 'smoker'])['charges'].mean().unstack()
smoker_region['Ratio (yes/no)'] = smoker_region['yes'] / smoker_region['no']
print(smoker_region)

plt.figure(figsize=(8, 5))
sns.barplot(data=df, x='region', y='charges', hue='smoker', ci=None)
plt.title('Charges by Region and Smoking Status')
plt.show()

## Câu hỏi 2: BMI có tương quan với chi phí mạnh hơn ở nhóm hút thuốc hay không hút thuốc?

In [ ]:
# TODO
corr_smoker = df[df['smoker'] == 'yes']['bmi'].corr(df[df['smoker'] == 'yes']['charges'])
corr_non_smoker = df[df['smoker'] == 'no']['bmi'].corr(df[df['smoker'] == 'no']['charges'])

print(f"Tương quan BMI & Charges (Hút thuốc): {corr_smoker:.3f}")
print(f"Tương quan BMI & Charges (Không hút thuốc): {corr_non_smoker:.3f}")

sns.lmplot(data=df, x='bmi', y='charges', hue='smoker', aspect=1.4)
plt.title('BMI vs Charges by Smoker')
plt.show()

## Câu hỏi 3: Vùng nào có chi phí bảo hiểm trung bình cao nhất?

In [ ]:
# TODO
region_charges = df.groupby('region')['charges'].mean().sort_values(ascending=False)
print(region_charges)
sns.barplot(x=region_charges.index, y=region_charges.values)
plt.title('Average Charges by Region')
plt.show()

## Câu hỏi 4: Số lượng con cái có làm tăng chi phí bảo hiểm không?

In [ ]:
# TODO
print(df.groupby('children')['charges'].agg(['count', 'mean', 'median']))
sns.boxplot(data=df, x='children', y='charges')
plt.title('Charges by Number of Children')
plt.show()

## Câu hỏi 5: Tuổi có tương quan với chi phí bảo hiểm không?

In [ ]:
# TODO
corr_age = df['age'].corr(df['charges'])
print(f"Hệ số tương quan giữa Age và Charges: {corr_age:.3f}")

sns.scatterplot(data=df, x='age', y='charges', hue='smoker', alpha=0.7)
plt.title('Age vs Charges')
plt.show()

## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể.

*(Viết insight của bạn vào đây...)*